<a href="https://colab.research.google.com/github/aymensrihi/deep-learning-projects/blob/main/possiblecvt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
CVT (Convolutional Vision Transformer) with SPSD for Domain Generalization
Paper: "CvT: Introducing Convolutions to Vision Transformers" + SPSD
Complete implementation for Diabetic Retinopathy classification
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import numpy as np
import random
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import math
from tqdm import tqdm

# ============================================================================
# CVT COMPONENTS - Convolutional Token Embeddings
# ============================================================================

def trunc_normal_(tensor, mean=0., std=1., a=-2., b=2.):
    """Truncated normal initialization"""
    def norm_cdf(x):
        return (1. + math.erf(x / math.sqrt(2.))) / 2.

    with torch.no_grad():
        l = norm_cdf((a - mean) / std)
        u = norm_cdf((b - mean) / std)
        tensor.uniform_(2 * l - 1, 2 * u - 1)
        tensor.erfinv_()
        tensor.mul_(std * math.sqrt(2.))
        tensor.add_(mean)
        tensor.clamp_(min=a, max=b)
        return tensor

class ConvEmbed(nn.Module):
    """Convolutional Token Embedding - Key CVT innovation"""
    def __init__(self, in_chans=3, embed_dim=64, stride=4, padding=2, norm_layer=None):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_chans, embed_dim // 2, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(embed_dim // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(embed_dim // 2, embed_dim, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(embed_dim, embed_dim, kernel_size=3, stride=1, padding=1, bias=False),
        )
        self.norm = norm_layer(embed_dim) if norm_layer else None

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)  # (B, H*W, C)
        if self.norm:
            x = self.norm(x)
        return x, (H, W)

class ConvProjection(nn.Module):
    """Convolutional Projection for Q, K, V - CVT innovation"""
    def __init__(self, dim, kernel_size=3, stride=1, padding=1, method='dw_bn'):
        super().__init__()
        self.stride = stride
        self.method = method

        if method == 'dw_bn':
            self.conv = nn.Sequential(
                nn.Conv2d(dim, dim, kernel_size=kernel_size, stride=stride,
                         padding=padding, groups=dim, bias=False),
                nn.BatchNorm2d(dim),
            )
        elif method == 'avg':
            self.conv = nn.AvgPool2d(kernel_size=kernel_size, stride=stride, padding=padding)
        else:
            raise ValueError(f"Unknown method: {method}")

    def forward(self, x, H, W):
        B, N, C = x.shape
        # Reshape to spatial
        x = x.transpose(1, 2).reshape(B, C, H, W)
        x = self.conv(x)
        _, _, H_new, W_new = x.shape
        x = x.flatten(2).transpose(1, 2)
        return x, H_new, W_new

class ConvAttention(nn.Module):
    """Convolutional Attention - CVT's key component"""
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.,
                 qkv_method='dw_bn', kernel_size=3, stride_q=1, stride_kv=1, padding_q=1, padding_kv=1):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5

        # Convolutional projections for Q, K, V
        self.conv_proj_q = ConvProjection(dim, kernel_size, stride_q, padding_q, qkv_method)
        self.conv_proj_k = ConvProjection(dim, kernel_size, stride_kv, padding_kv, qkv_method)
        self.conv_proj_v = ConvProjection(dim, kernel_size, stride_kv, padding_kv, qkv_method)

        self.proj_q = nn.Linear(dim, dim, bias=qkv_bias)
        self.proj_k = nn.Linear(dim, dim, bias=qkv_bias)
        self.proj_v = nn.Linear(dim, dim, bias=qkv_bias)

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, H, W):
        B, N, C = x.shape

        # Apply convolutional projections
        q, H_q, W_q = self.conv_proj_q(x, H, W)
        k, H_k, W_k = self.conv_proj_k(x, H, W)
        v, H_v, W_v = self.conv_proj_v(x, H, W)

        # Linear projections
        q = self.proj_q(q).reshape(B, H_q * W_q, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        k = self.proj_k(k).reshape(B, H_k * W_k, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        v = self.proj_v(v).reshape(B, H_v * W_v, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)

        # Attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, H_q * W_q, C)
        x = self.proj(x)
        x = self.proj_drop(x)

        return x, H_q, W_q

class Mlp(nn.Module):
    """MLP with GELU activation"""
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class ConvBlock(nn.Module):
    """CVT Block with Convolutional Attention"""
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, drop=0., attn_drop=0.,
                 qkv_method='dw_bn', kernel_size=3, stride_q=1, stride_kv=1, padding_q=1, padding_kv=1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = ConvAttention(
            dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop,
            qkv_method=qkv_method, kernel_size=kernel_size, stride_q=stride_q, stride_kv=stride_kv,
            padding_q=padding_q, padding_kv=padding_kv
        )
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, drop=drop)

    def forward(self, x, H, W):
        x_attn, H_new, W_new = self.attn(self.norm1(x), H, W)
        x = x[:, :H_new*W_new] + x_attn  # Residual connection
        x = x + self.mlp(self.norm2(x))
        return x, H_new, W_new

class ConvStage(nn.Module):
    """CVT Stage - multiple ConvBlocks"""
    def __init__(self, embed_dim, num_heads, depth, mlp_ratio=4., qkv_bias=False, drop=0., attn_drop=0.,
                 qkv_method='dw_bn', kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.blocks = nn.ModuleList([
            ConvBlock(
                dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias,
                drop=drop, attn_drop=attn_drop, qkv_method=qkv_method, kernel_size=kernel_size,
                stride_q=1, stride_kv=stride, padding_q=padding, padding_kv=padding
            ) for _ in range(depth)
        ])

    def forward(self, x, H, W):
        for blk in self.blocks:
            x, H, W = blk(x, H, W)
        return x, H, W

# ============================================================================
# CVT MODEL with SPSD
# ============================================================================

class CVT(nn.Module):
    """Convolutional Vision Transformer with SPSD support"""
    def __init__(self, img_size=224, in_chans=3, num_classes=1000,
                 embed_dims=[64, 192, 384], depths=[1, 4, 16], num_heads=[1, 3, 6],
                 mlp_ratios=[4, 4, 4], qkv_bias=True, drop_rate=0., attn_drop_rate=0.):
        super().__init__()
        self.num_classes = num_classes
        self.num_stages = len(embed_dims)

        # Stage 1: Convolutional Embedding
        self.patch_embed1 = ConvEmbed(in_chans=in_chans, embed_dim=embed_dims[0],
                                      stride=4, padding=2, norm_layer=nn.LayerNorm)

        # Stage 2: Convolutional Embedding
        self.patch_embed2 = ConvEmbed(in_chans=embed_dims[0], embed_dim=embed_dims[1],
                                      stride=2, padding=1, norm_layer=nn.LayerNorm)

        # Stage 3: Convolutional Embedding
        self.patch_embed3 = ConvEmbed(in_chans=embed_dims[1], embed_dim=embed_dims[2],
                                      stride=2, padding=1, norm_layer=nn.LayerNorm)

        # CVT Stages
        self.stage1 = ConvStage(embed_dims[0], num_heads[0], depths[0], mlp_ratios[0],
                               qkv_bias, drop_rate, attn_drop_rate, kernel_size=3, stride=2, padding=1)

        self.stage2 = ConvStage(embed_dims[1], num_heads[1], depths[1], mlp_ratios[1],
                               qkv_bias, drop_rate, attn_drop_rate, kernel_size=3, stride=2, padding=1)

        self.stage3 = ConvStage(embed_dims[2], num_heads[2], depths[2], mlp_ratios[2],
                               qkv_bias, drop_rate, attn_drop_rate, kernel_size=3, stride=2, padding=1)

        # Classification heads for each stage (SPSD)
        self.norm1 = nn.LayerNorm(embed_dims[0])
        self.norm2 = nn.LayerNorm(embed_dims[1])
        self.norm3 = nn.LayerNorm(embed_dims[2])

        self.head1 = nn.Linear(embed_dims[0], num_classes)
        self.head2 = nn.Linear(embed_dims[1], num_classes)
        self.head3 = nn.Linear(embed_dims[2], num_classes)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward_features(self, x):
        """Returns features from all stages for SPSD"""
        B = x.shape[0]
        outputs = []

        # Stage 1
        x, (H, W) = self.patch_embed1(x)
        x, H, W = self.stage1(x, H, W)
        x = self.norm1(x)
        feat1 = x.mean(dim=1)  # Global average pooling
        outputs.append(feat1)

        # Reshape for next stage
        x = x.transpose(1, 2).reshape(B, -1, H, W)

        # Stage 2
        x, (H, W) = self.patch_embed2(x)
        x, H, W = self.stage2(x, H, W)
        x = self.norm2(x)
        feat2 = x.mean(dim=1)
        outputs.append(feat2)

        # Reshape for next stage
        x = x.transpose(1, 2).reshape(B, -1, H, W)

        # Stage 3
        x, (H, W) = self.patch_embed3(x)
        x, H, W = self.stage3(x, H, W)
        x = self.norm3(x)
        feat3 = x.mean(dim=1)
        outputs.append(feat3)

        return outputs

    def forward(self, x):
        """Returns predictions from all stages"""
        features = self.forward_features(x)
        outputs = [
            self.head1(features[0]),
            self.head2(features[1]),
            self.head3(features[2])
        ]
        return outputs

# ============================================================================
# DATASET CLASS
# ============================================================================

class DRDataset(Dataset):
    """Diabetic Retinopathy Dataset"""
    def __init__(self, root, domain_name, transform=None):
        self.root = os.path.join(root, domain_name)
        self.transform = transform
        self.classes = ['0', '1', '2', '3', '4']
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        self.samples = []
        image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.gif')

        for class_name in self.classes:
            class_dir = os.path.join(self.root, class_name)
            if not os.path.exists(class_dir):
                continue

            class_idx = self.class_to_idx[class_name]
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(image_extensions):
                    self.samples.append((os.path.join(class_dir, img_name), class_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, target = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, target
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return self.__getitem__(random.randint(0, len(self) - 1))

# ============================================================================
# CVT-SPSD ALGORITHM
# ============================================================================

class CVT_SPSD(nn.Module):
    """CVT with Soft Prediction Self-Distillation"""
    def __init__(self, num_classes=5, hparams=None):
        super().__init__()
        if hparams is None:
            hparams = default_hparams()

        self.hparams = hparams
        self.lambda_ = hparams['RB_loss_weight']
        self.beta_T = hparams['alpha_T']
        self.n_steps = hparams['n_steps']
        self.step_count = 0
        self.n_classes = num_classes

        # CVT model (3 stages like CVT-13)
        self.network = CVT(
            img_size=224,
            in_chans=3,
            num_classes=num_classes,
            embed_dims=[64, 192, 384],
            depths=[1, 2, 10],
            num_heads=[1, 3, 6],
            mlp_ratios=[4, 4, 4],
            qkv_bias=True,
            drop_rate=0.,
            attn_drop_rate=0.
        )

        self.optimizer = torch.optim.AdamW(
            self.network.parameters(),
            lr=hparams["lr"],
            weight_decay=hparams['weight_decay']
        )

    def update(self, x, y):
        """Training step with SPSD"""
        # Progressive beta_t
        beta_t = self.beta_T * ((self.step_count + 1) / self.n_steps)
        beta_t = max(0.0, min(beta_t, self.beta_T))
        self.step_count += 1

        # Forward pass - get outputs from all stages
        outputs = self.network(x)
        z = outputs[-1]  # Final stage output

        # Random stage selection
        stage_idx = random.randint(0, len(outputs) - 1)
        z_j = outputs[stage_idx]

        # One-hot labels
        y_one_hot = torch.zeros(y.size(0), self.n_classes, device=x.device)
        y_one_hot.scatter_(1, y.unsqueeze(1), 1)

        # Progressive Soft Pseudo Labeling
        soft_z = beta_t * z + (1 - beta_t) * y_one_hot
        soft_z_j = beta_t * z_j + (1 - beta_t) * y_one_hot

        soft_output = F.softmax(soft_z, dim=1)
        soft_output_stage = F.softmax(soft_z_j, dim=1)

        # Losses
        base_loss = F.cross_entropy(z, y)
        distill_loss = F.kl_div(
            torch.log(soft_output_stage + 1e-10),
            soft_output.detach(),
            reduction='batchmean',
            log_target=False
        )

        loss = base_loss + self.lambda_ * distill_loss

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return {
            'loss': loss.item(),
            'base_loss': base_loss.item(),
            'distill_loss': distill_loss.item(),
            'beta_t': beta_t
        }

    def predict(self, x):
        outputs = self.network(x)
        return outputs[-1]  # Return final stage output

# ============================================================================
# TRAINING UTILITIES
# ============================================================================

def default_hparams():
    return {
        'data_augmentation': True,
        'RB_loss_weight': 0.7,
        'KL_Div_Temperature': 4.0,
        'alpha_T': 0.8,
        'n_steps': 5000,
        'lr': 5e-5,
        'weight_decay': 0.0,
        'batch_size': 32
    }

def get_transforms(augment=True):
    transform_list = [transforms.Resize((224, 224))]

    if augment:
        transform_list.extend([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.4, contrast=0.4)
        ])

    transform_list.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    return transforms.Compose(transform_list)

@torch.no_grad()
def evaluate(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0

    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        predictions = model.predict(x)
        _, predicted = torch.max(predictions, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)

    return correct / total if total > 0 else 0.0

def train_multi_source_dg(data_root, test_domain, num_epochs=10, hparams=None):
    if hparams is None:
        hparams = default_hparams()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}\n")

    all_domains = ['aptos', 'eyepacs', 'messidor', 'messidor_2']
    train_domains = [d for d in all_domains if d != test_domain]

    print(f"Training domains: {train_domains}")
    print(f"Test domain: {test_domain}\n")

    train_transform = get_transforms(augment=True)
    test_transform = get_transforms(augment=False)

    train_datasets = [DRDataset(data_root, d, train_transform) for d in train_domains]
    test_dataset = DRDataset(data_root, test_domain, test_transform)

    train_loaders = [
        DataLoader(ds, batch_size=hparams['batch_size'], shuffle=True, num_workers=2)
        for ds in train_datasets
    ]
    test_loader = DataLoader(test_dataset, batch_size=hparams['batch_size'],
                            shuffle=False, num_workers=2)

    print(f"Training samples: {sum(len(ds) for ds in train_datasets)}")
    print(f"Test samples: {len(test_dataset)}\n")

    model = CVT_SPSD(num_classes=5, hparams=hparams).to(device)

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}\n")

    best_acc = 0.0

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        epoch_base_loss = 0
        epoch_distill_loss = 0
        n_batches = 0

        train_iters = [iter(loader) for loader in train_loaders]
        max_batches = max(len(loader) for loader in train_loaders)

        pbar = tqdm(range(max_batches), desc=f"Epoch {epoch+1}/{num_epochs}")

        for batch_idx in pbar:
            all_x, all_y = [], []
            for train_iter, loader in zip(train_iters, train_loaders):
                try:
                    x, y = next(train_iter)
                except StopIteration:
                    train_iter = iter(loader)
                    x, y = next(train_iter)
                all_x.append(x)
                all_y.append(y)

            x = torch.cat(all_x).to(device)
            y = torch.cat(all_y).to(device)

            step_vals = model.update(x, y)

            epoch_loss += step_vals['loss']
            epoch_base_loss += step_vals['base_loss']
            epoch_distill_loss += step_vals['distill_loss']
            n_batches += 1

            pbar.set_postfix({
                'loss': f"{step_vals['loss']:.4f}",
                'β_t': f"{step_vals['beta_t']:.4f}"
            })

        test_acc = evaluate(model, test_loader, device)

        print(f"\nEpoch {epoch+1}/{num_epochs}:")
        print(f"  Loss: {epoch_loss/n_batches:.4f}")
        print(f"  Base Loss: {epoch_base_loss/n_batches:.4f}")
        print(f"  Distill Loss: {epoch_distill_loss/n_batches:.4f}")
        print(f"  Test Accuracy ({test_domain}): {test_acc*100:.2f}%")

        if test_acc > best_acc:
            best_acc = test_acc
            print(f"  ✓ New best accuracy!")
        print()

    print(f"Training completed! Best test accuracy: {best_acc*100:.2f}%")
    return model, best_acc

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("=" * 80)
    print("CVT-SPSD: Convolutional Vision Transformer with Self-Distillation")
    print("Diabetic Retinopathy Domain Generalization")
    print("=" * 80)

    data_path = "/content/DR"

    if not os.path.exists(data_path):
        print(f"ERROR: Dataset not found at {data_path}")
        exit(1)

    print(f"\n✓ Dataset found at: {data_path}")
    print(f"✓ Domains available: {sorted(os.listdir(data_path))}\n")

    test_domain = 'messidor_2'
    num_epochs = 10

    hparams = default_hparams()
    hparams['n_steps'] = 5000

    print(f"Configuration:")
    print(f"  Architecture: CVT-13 (Convolutional ViT)")
    print(f"  Test domain: {test_domain}")
    print(f"  Epochs: {num_epochs}")
    print(f"  Batch size: {hparams['batch_size']}")
    print(f"  Learning rate: {hparams['lr']}")
    print(f"  λ (distillation weight): {hparams['RB_loss_weight']}")
    print(f"  β_T (max mixing): {hparams['alpha_T']}\n")

    model, best_acc = train_multi_source_dg(
        data_root=data_path,
        test_domain=test_domain,
        num_epochs=num_epochs,
        hparams=hparams
    )

    print("\n" + "=" * 80)
    print(f"✓ Training complete!")
    print(f"✓ Best accuracy on {test_domain}: {best_acc*100:.2f}%")
    print("=" * 80)

CVT-SPSD: Convolutional Vision Transformer with Self-Distillation
Diabetic Retinopathy Domain Generalization
ERROR: Dataset not found at /content/DR

✓ Dataset found at: /content/DR


FileNotFoundError: [Errno 2] No such file or directory: '/content/DR'